In [3]:
import requests
from typing import Dict
import pandas as pd
from dotenv import load_dotenv
import os
import json
from datetime import datetime

In [4]:
BASE_URL = "http://localhost:8000"

In [5]:
MAX_ID_FILE = "max_ids.json"


def load_max_ids():
    if not os.path.exists(MAX_ID_FILE):
        with open(MAX_ID_FILE, "w") as f:
            json.dump({}, f)
        return {}

    with open(MAX_ID_FILE, "r") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            return {}


def save_max_ids(max_ids: Dict[str, int]):
    with open(MAX_ID_FILE, "w") as f:
        json.dump(max_ids, f, indent=4)


def fetch_all_pages(endpoint: str, page_size: int = 10000):
    all_data = []
    page = 1

    max_ids = load_max_ids()

    last_max_id = max_ids.get(endpoint)

    while True:
        params = {
            "page_size": page_size,
        }

        if last_max_id is not None:
            params["last_max_id"] = last_max_id

        response = requests.get(
            f"{BASE_URL}{endpoint}",
            params=params,
            timeout=120
        )

        if response.status_code != 200:
            print(f"Error {response.status_code}: {response.text}")
            break

        result = response.json()
        items = result.get("data", [])

        if not items:
            break

        all_data.extend(items)
        print(f"Fetched page {page} → {len(items)} records")

        last_item = items[-1]

        if "id" in last_item:
            last_max_id = last_item["id"]
        elif "user_id" in last_item:
            last_max_id = last_item["user_id"]

        if len(items) < page_size:
            break

        page += 1

    if last_max_id is not None:
        max_ids[endpoint] = last_max_id
        save_max_ids(max_ids)

    df = pd.DataFrame(all_data)
    print(f"Total records fetched: {len(all_data)}")
    return df

In [6]:
from io import BytesIO
import math
import pandas as pd
from azure.storage.blob import BlobServiceClient, BlobBlock


class AzureCSVUploader:
    def __init__(self, connection_string: str, container_name: str):
        self.blob_service_client = BlobServiceClient.from_connection_string(
            connection_string,
            max_single_put_size=4 * 1024 * 1024,    
            max_block_size=4 * 1024 * 1024,           
        )
        self.container_name = container_name

        container_client = self.blob_service_client.get_container_client(
            container_name
        )

        try:
            container_client.create_container()
        except Exception:
            pass

    ROWS_PER_FILE = 100_000

    def upload_df(
        self,
        df: pd.DataFrame,
        blob_name: str,
        timeout: int = 600,
        max_concurrency: int = 4,
    ):
        if df.empty:
            print(f"Skipping empty DataFrame → {blob_name}")
            return

        csv_buffer = BytesIO()
        df.to_csv(csv_buffer, index=False, encoding="utf-8")
        size_mb = csv_buffer.tell() / 1024 / 1024
        csv_buffer.seek(0)

        blob_client = self.blob_service_client.get_blob_client(
            container=self.container_name,
            blob=blob_name,
        )

        for attempt in range(3):
            try:
                blob_client.upload_blob(
                    csv_buffer,
                    overwrite=True,
                    timeout=timeout,
                    max_concurrency=max_concurrency,
                )
                print(f"Uploaded ({size_mb:.1f} MB) → {blob_name}")
                return

            except Exception as e:
                print(f"Upload failed (attempt {attempt + 1}/3) → {blob_name}: {e}")
                csv_buffer.seek(0)

        raise RuntimeError(f"Failed after 3 attempts → {blob_name}")

    def upload_df_in_chunks(
        self,
        df: pd.DataFrame,
        base_blob_path: str,
        timeout: int = 600,
        max_concurrency: int = 4,
    ):
        if df.empty:
            print(f"Skipping empty DataFrame → {base_blob_path}")
            return

        total_rows = len(df)
        total_parts = math.ceil(total_rows / self.ROWS_PER_FILE)

        for part_num, start in enumerate(
            range(0, total_rows, self.ROWS_PER_FILE), start=1
        ):
            chunk_df = df.iloc[start : start + self.ROWS_PER_FILE]
            blob_name = f"{base_blob_path}/part_{part_num:03d}_of_{total_parts:03d}.csv"
            self.upload_df(chunk_df, blob_name, timeout=timeout, max_concurrency=max_concurrency)

        print(f"Finished uploading {total_rows:,} rows in {total_parts} files → {base_blob_path}")

In [7]:
load_dotenv()
CONNECTION_STRING = os.getenv("CONNECTION_STRING")
CONTAINER_NAME = os.getenv("CONTAINER_NAME")
uploader = AzureCSVUploader(connection_string = CONNECTION_STRING, container_name = CONTAINER_NAME)

date_folder = datetime.utcnow().strftime("%Y-%m-%d")

students = fetch_all_pages("/export_students")
courses = fetch_all_pages("/export_courses")
enrollments = fetch_all_pages("/export_enrollments")
attendance = fetch_all_pages("/export_attendance")
course_offerings = fetch_all_pages("/export_course_offerings")
department = fetch_all_pages("/export_department")
academic_semester = fetch_all_pages("/export_academic_semester")
tickets = fetch_all_pages("/export_tickets")


uploader.upload_df(students, f"students/{date_folder}/students.csv")
uploader.upload_df(courses, f"courses/{date_folder}/courses.csv")
uploader.upload_df(enrollments, f"enrollments/{date_folder}/enrollments.csv")
uploader.upload_df(attendance, f"attendance/{date_folder}/attendance.csv")
uploader.upload_df(course_offerings, f"course_offerings/{date_folder}/course_offerings.csv")
uploader.upload_df(department, f"department/{date_folder}/department.csv")
uploader.upload_df(academic_semester, f"academic_semester/{date_folder}/academic_semester.csv")
uploader.upload_df(tickets, f"tickets/{date_folder}/tickets.csv")

Total records fetched: 0
Total records fetched: 0
Total records fetched: 0
Fetched page 1 → 10000 records
Fetched page 2 → 10000 records
Fetched page 3 → 10000 records
Fetched page 4 → 10000 records
Fetched page 5 → 10000 records
Fetched page 6 → 10000 records
Fetched page 7 → 10000 records
Fetched page 8 → 10000 records
Fetched page 9 → 10000 records
Fetched page 10 → 10000 records
Fetched page 11 → 10000 records
Fetched page 12 → 10000 records
Fetched page 13 → 10000 records
Fetched page 14 → 10000 records
Fetched page 15 → 10000 records
Fetched page 16 → 10000 records
Fetched page 17 → 10000 records
Fetched page 18 → 10000 records
Fetched page 19 → 10000 records
Fetched page 20 → 10000 records
Fetched page 21 → 10000 records
Fetched page 22 → 10000 records
Fetched page 23 → 10000 records
Fetched page 24 → 10000 records
Fetched page 25 → 10000 records
Fetched page 26 → 10000 records
Fetched page 27 → 10000 records
Fetched page 28 → 10000 records
Fetched page 29 → 10000 records
Fetche